# SCPA SBERT Notebook

This notebook is self-contained and runnable without downloading model weights. For production or offline fine-tuning, the recommended pretrained model is `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` because it supports Indonesian and English text with 384-dimensional sentence embeddings.

## Profile and Job Encoding Formula

Fallback encoding maps text to a normalized vector:

`embedding(text) = normalize(category_features(text) + hashed_token_features(text))`

Fallback scoring combines category overlap, lexical overlap, and cosine similarity:

`score(profile, job) = clamp(0.12 + 0.62*category_overlap + 0.16*lexical_overlap + 0.10*cosine01, 0, 1)`

In [1]:
import hashlib
import math
import re
import numpy as np

EMBEDDING_DIM = 384
TOKEN_RE = re.compile(r"[a-z0-9+#.]+")
ALIASES = {
    "communication": {"communication", "presenter", "presentation", "public", "speaking", "speaker", "host", "hosting", "mc", "master", "ceremony", "emcee", "moderator"},
    "language": {"english", "inggris", "sastra", "literature", "translator", "translation", "writing", "content"},
    "event": {"event", "ceremony", "wedding", "seminar", "conference", "workshop", "audience"},
    "software": {"backend", "frontend", "developer", "engineer", "python", "java", "javascript", "fastapi", "api", "database", "postgres", "react", "node"},
    "data": {"data", "machine", "learning", "ml", "ai", "analytics", "scientist", "pandas", "numpy", "model"},
    "business": {"sales", "marketing", "business", "account", "customer", "growth"},
    "design": {"ui", "ux", "designer", "figma", "prototype", "visual", "product"},
}
NORMALISE = {"bahasa": "language", "inggris": "english", "pembawa": "host", "acara": "event", "pewara": "emcee", "komunikasi": "communication"}

def tokens(text):
    return [NORMALISE.get(token, token) for token in TOKEN_RE.findall(text.lower())]

def category_scores(token_values):
    token_set = set(token_values)
    return {name: min(1.0, len(token_set & aliases) / 3.0) for name, aliases in ALIASES.items()}

def deterministic_embedding(text, dim=EMBEDDING_DIM):
    token_values = tokens(text)
    token_set = set(token_values)
    vec = np.zeros(dim, dtype=np.float32)
    cats = category_scores(token_values)
    for i, name in enumerate(ALIASES):
        vec[i] = cats[name]
    for token in token_set:
        digest = hashlib.sha256(token.encode("utf-8")).digest()
        idx = 32 + int.from_bytes(digest[:2], "big") % (dim - 32)
        sign = 1.0 if digest[2] % 2 == 0 else -1.0
        vec[idx] += sign * (0.08 + digest[3] / 2550.0)
    if not token_set:
        vec[31] = 1.0
    return vec / (np.linalg.norm(vec) + 1e-12)

def cosine(left, right):
    return float(np.dot(left, right) / ((np.linalg.norm(left) * np.linalg.norm(right)) + 1e-12))

def sbert_score(profile, job):
    left_tokens, right_tokens = set(tokens(profile)), set(tokens(job))
    if not left_tokens or not right_tokens:
        return 0.2
    left_categories, right_categories = category_scores(left_tokens), category_scores(right_tokens)
    overlap = total = 0.0
    for name in ALIASES:
        weight = 1.35 if name in {"communication", "language", "event", "software"} else 1.0
        overlap += weight * min(left_categories[name], right_categories[name])
        total += weight * max(left_categories[name], right_categories[name])
    category_overlap = overlap / total if total else 0.0
    lexical_overlap = len(left_tokens & right_tokens) / math.sqrt(max(1, len(left_tokens)) * max(1, len(right_tokens)))
    cosine01 = (cosine(deterministic_embedding(profile), deterministic_embedding(job)) + 1.0) / 2.0
    score = 0.12 + 0.62 * category_overlap + 0.16 * lexical_overlap + 0.10 * cosine01
    if left_categories["software"] and right_categories["software"]:
        score += 0.08
    if (left_categories["communication"] or left_categories["language"]) and (right_categories["communication"] or right_categories["event"]) and not right_categories["software"]:
        score += 0.24
    if (left_categories["communication"] or left_categories["language"]) and right_categories["software"] and not (left_categories["software"] or right_categories["communication"]):
        score -= 0.18
    return round(float(min(1.0, max(0.0, score))), 4)


## Cosine Demo

In [2]:
profile = "Sastra Inggris Public Speaking"
mc_job = "Master of Ceremony"
backend_job = "Backend Developer"

print("cosine(profile, MC):", round(cosine(deterministic_embedding(profile), deterministic_embedding(mc_job)), 4))
print("cosine(profile, Backend):", round(cosine(deterministic_embedding(profile), deterministic_embedding(backend_job)), 4))
print("sbert_score(profile, MC):", sbert_score(profile, mc_job))
print("sbert_score(profile, Backend):", sbert_score(profile, backend_job))


cosine(profile, MC): 0.5897
cosine(profile, Backend): 0.0
sbert_score(profile, MC): 0.6875
sbert_score(profile, Backend): 0.0


## Sastra Inggris to MC High, Backend Developer Low

In [3]:
scores = {
    "Master of Ceremony": sbert_score("Sastra Inggris Public Speaking", "Master of Ceremony"),
    "Backend Developer": sbert_score("Sastra Inggris Public Speaking", "Backend Developer"),
}
scores


{'Master of Ceremony': 0.6875, 'Backend Developer': 0.0}

## Assertion Block

In [4]:
assert sbert_score("Sastra Inggris Public Speaking", "Master of Ceremony") > 0.6
assert sbert_score("Sastra Inggris Public Speaking", "Backend Developer") < 0.4
assert deterministic_embedding("hello").shape == (384,)
print("All SBERT notebook assertions passed.")


All SBERT notebook assertions passed.
